# Step 4: Prompted LLM (Qwen2.5-3B-Instruct) on Colab

**Author:** Michael Dong  
**Project:** PlainMed (COSC 72/272 final project)  
**Date:** November 14, 2026

**What this notebook does:** runs the prompted-LLM systems on a free Colab T4 GPU:
1. a smoke test (2 examples) that also measures speed,
2. a one-off check that real BERTScore works (Step 2 left this untested),
3. the prompt-sensitivity run: prompt wordings A, B, C × {zero-shot, 3-shot} on the same 100 validation examples,
4. the prompt choice, made by the rule fixed in `DECISIONS.md` before any results existed,
5. the chosen prompt on all 411 validation examples, zero-shot and 3-shot.

**Input:** the `dongm58/plainmed` GitHub repo (it rebuilds the data itself).  
**Output:** everything is written straight to Google Drive in `MyDrive/plainmed/step4/` (outputs and results), so a disconnect loses at most one batch. Re-running a cell continues where it stopped.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. If the repo is private, add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar) and turn on notebook access for it.

### 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Mount Google Drive (outputs are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/plainmed/step4'
import os
os.makedirs(RUN_DIR, exist_ok=True)
print('outputs will be saved in', RUN_DIR)

### 3. Get the code (uses `GH_TOKEN` only if the repo is private)

In [ ]:
import subprocess
def clone():
    url = 'https://github.com/dongm58/plainmed'
    if subprocess.run(['git', 'clone', '-q', '--depth', '1', url, '/content/plainmed']).returncode == 0:
        return
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    subprocess.run(['git', 'clone', '-q', '--depth', '1', f'https://{token}@github.com/dongm58/plainmed', '/content/plainmed'], check=True)
if not os.path.exists('/content/plainmed'):
    clone()
%cd /content/plainmed
!git log -1 --format='commit %h: %s'

### 4. Install the project (Colab already has PyTorch, so it is not reinstalled)

In [ ]:
!pip -q install -r requirements-colab.txt && pip -q install -e . --no-deps
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"

### 5. Build the data (about 30 seconds)

In [ ]:
!python scripts/01_data.py > /tmp/data_log.txt && head -2 /tmp/data_log.txt

### 6. Smoke test
Prints two outputs for each setting and the seconds per example. **Read the outputs:** if they are empty, garbled or repeat themselves, half precision (float16) is misbehaving. Re-run this cell with `--dtype float32`, and add the same flag to the later cells.

In [ ]:
!python scripts/04_llm.py smoke

### 7. One-off check that real BERTScore works (downloads roberta-large, about 1.4 GB)

In [ ]:
import sys
sys.path.insert(0, '/content/plainmed')
from plainmed.io import read_jsonl
from plainmed.metrics import score_system
val = read_jsonl('data/processed/cochrane/val.jsonl')[:3]
outputs = [{'id': e['id'], 'output': e['source']} for e in val]
print(score_system(outputs, val, bertscore='real', bertscore_model='roberta-large')[['id', 'sari', 'rouge_l', 'bertscore_f1']])

### 8. Prompt sensitivity: 3 wordings × 2 settings × 100 validation examples
Saves to Drive as it goes. If Colab disconnects, run cells 2–5 again and then this one: finished examples are skipped.

In [ ]:
!python scripts/04_llm.py sensitivity --out-root $RUN_DIR --batch-size 8

### 9. Choose the prompt (rule fixed in advance in DECISIONS.md)

In [ ]:
!python scripts/04_llm.py choose --out-root $RUN_DIR
import json
VARIANT = json.load(open(f'{RUN_DIR}/results/prompt_choice.json'))['chosen_variant']
print('chosen variant:', VARIANT)

### 10. Full validation runs with the chosen prompt (411 examples each)

In [ ]:
!python scripts/04_llm.py run --shots zero --variant $VARIANT --out-root $RUN_DIR --batch-size 8

In [ ]:
!python scripts/04_llm.py run --shots few --variant $VARIANT --out-root $RUN_DIR --batch-size 8

### 11. Check what was saved
All files are in `MyDrive/plainmed/step4/`. Tell Claude when this cell shows 411 lines for both systems.

In [ ]:
!wc -l $RUN_DIR/outputs/*/*.jsonl
!head -c 600 $RUN_DIR/results/prompt_choice.json